# 📊 Benchmark ChatBDI — Confronto Modelli
Benchmark completo per confrontare modelli base e fine-tuned sulla generazione di literal AgentSpeak in formato JSON.

In [ ]:
# ==========================================
# 0. INSTALLAZIONE DIPENDENZE
# ==========================================
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes -q

## 🚀 Benchmark Principale (Base + Fine-Tuned)
Esegue inferenza su Qwen2.5-Coder-7B (Base) e Qwen3.5-4B (Fine-Tuned), calcola Exact Match e Slot-Filling Accuracy, e salva i risultati in CSV.

In [ ]:
import json
import torch
import re
import time
import pandas as pd
import gc
import os
from tqdm import tqdm
from transformers import StoppingCriteria, StoppingCriteriaList
from unsloth import FastLanguageModel
from google.colab import drive

# ==========================================
# 1. MONTAGGIO DRIVE E PERCORSI
# ==========================================
drive.mount('/content/drive')
DRIVE_FOLDER = "/content/drive/MyDrive/Tirocinio_bechelor/"
os.makedirs(DRIVE_FOLDER, exist_ok=True)
PATH_MODELLO_FINETUNED = os.path.join(DRIVE_FOLDER, "lora_chatbdi_qwen3.5-4B")

# ==========================================
# 2. FUNZIONI DI SUPPORTO
# ==========================================
def normalize(obj):
    if not isinstance(obj, dict):
        return obj
    out = {}
    for k, v in obj.items():
        if isinstance(v, str):
            v = v.strip()
            if v == "_" or (v and v[0].isupper() and v.replace("_", "").isalpha()):
                v = "__VAR__"
        out[k] = v
    return out

def fix_hallucinated_json(text):
    text = re.sub(r'([a-zA-Z0-9_]+)\s*=', r'"\1": ', text)
    text = re.sub(r'([{,]\s*)([a-zA-Z0-9_]+)(\s*:)', r'\1"\2"\3', text)
    text = re.sub(r'("arg\d+":\s*)"([^"]*)"', r'\1"\\"\2\\""', text)
    text = re.sub(r':\s*([^"\d\s\[{][^,}]*?)\s*([,}])', r': "\1"\2', text)
    return text

def calcola_accuratezza_parziale(expected_dict, predicted_dict):
    if not isinstance(expected_dict, dict) or not isinstance(predicted_dict, dict):
        return 0.0
    totale_chiavi = len(expected_dict)
    if totale_chiavi == 0:
        return 1.0 if len(predicted_dict) == 0 else 0.0
    chiavi_corrette = sum(1 for k, v in expected_dict.items() if k in predicted_dict and predicted_dict[k] == v)
    return chiavi_corrette / totale_chiavi

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set([s for s in stop_ids if s is not None])
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1].item() in self.stop_ids

# ==========================================
# 3. CONFIGURAZIONE ESPERIMENTI
# ==========================================
esperimenti = [
    {
        "nome_modello": "Qwen2.5-Coder-7B (Base)",
        "path": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
        "trained": "No",
        "datasets": [
            {"id": "base_ticket_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
            {"id": "base_ticket_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"},
            {"id": "base_ignoto_zero.jsonl", "domain": "In-Domain",     "esempi": "Zero-Shot"},
            {"id": "base_ignoto_few.jsonl",  "domain": "In-Domain",     "esempi": "Few-Shot"}
        ]
    },
    {
        "nome_modello": "Qwen3.5-4B (Fine-Tuned)",
        "path": PATH_MODELLO_FINETUNED,
        "trained": "Yes",
        "datasets": [
            {"id": "ft_ticket_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
            {"id": "ft_ticket_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"},
            {"id": "ft_ignoto_zero.jsonl", "domain": "In-Domain",     "esempi": "Zero-Shot"},
            {"id": "ft_ignoto_few.jsonl",  "domain": "In-Domain",     "esempi": "Few-Shot"}
        ]
    }
]

# ==========================================
# 4. INFERENZA MASSIVA E BENCHMARK
# ==========================================
risultati_globali = []

for config_mod in esperimenti:
    print(f"\n🚀 CARICAMENTO: {config_mod['nome_modello']}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = config_mod['path'],
        max_seq_length = 4096,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

    text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
    im_end_id = text_tokenizer.convert_tokens_to_ids("<|im_end|>")
    stop_crit = StoppingCriteriaList([StopOnTokens([text_tokenizer.eos_token_id, im_end_id])])

    for ds_config in config_mod['datasets']:
        file_path_drive = os.path.join(DRIVE_FOLDER, ds_config['id'])

        if not os.path.exists(file_path_drive):
            print(f"❌ SALTO: {ds_config['id']} non trovato su Drive.")
            continue

        with open(file_path_drive, encoding="utf-8") as f:
            test_cases = [json.loads(line) for line in f if line.strip()]

        for i, tc in enumerate(tqdm(test_cases, desc=f"Test {ds_config['id']}")):
            system = tc["messages"][0]["content"]
            user = tc["messages"][1]["content"]
            expected = tc["messages"][2]["content"]

            prompt_testuale = text_tokenizer.apply_chat_template(
                [{"role": "system", "content": system}, {"role": "user", "content": user}],
                tokenize=False, add_generation_prompt=False
            )
            prompt_testuale += "<|im_start|>assistant\n{"

            tokens = text_tokenizer(prompt_testuale, return_tensors="pt")
            input_ids = tokens["input_ids"].to("cuda")
            attention_mask = tokens["attention_mask"].to("cuda") if "attention_mask" in tokens else torch.ones_like(input_ids)
            prompt_len = input_ids.shape[1]

            start_time = time.time()
            with torch.no_grad():
                output_ids = model.generate(
                    input_ids = input_ids,
                    attention_mask = attention_mask,
                    max_new_tokens = 128,
                    do_sample = False,
                    pad_token_id = text_tokenizer.eos_token_id,
                    stopping_criteria = stop_crit,
                )
            end_time = time.time()

            latenza_sec = end_time - start_time
            token_generati = len(output_ids[0]) - prompt_len
            tps = token_generati / latenza_sec if latenza_sec > 0 else 0

            predicted_raw = text_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
            predicted_raw = re.sub(r'<think>.*?</think>', '', predicted_raw, flags=re.DOTALL).strip()
            predicted_raw = "{" + predicted_raw

            cleaned = predicted_raw
            backticks = "`" * 3
            if cleaned.startswith(backticks + "json"): cleaned = cleaned[7:]
            elif cleaned.startswith(backticks): cleaned = cleaned[3:]
            if cleaned.endswith(backticks): cleaned = cleaned[:-3]
            cleaned = cleaned.strip()

            try:
                json.loads(cleaned)
            except json.JSONDecodeError:
                cleaned = fix_hallucinated_json(cleaned)

            passed = False
            parziale = 0.0
            try:
                dict_atteso = normalize(json.loads(expected))
                dict_previsto = normalize(json.loads(cleaned))
                passed = (dict_atteso == dict_previsto)
                parziale = calcola_accuratezza_parziale(dict_atteso, dict_previsto)
            except json.JSONDecodeError:
                pass

            risultati_globali.append({
                "Modello": config_mod['nome_modello'],
                "Addestrato": config_mod['trained'],
                "Dominio": ds_config['domain'],
                "Strategia": ds_config['esempi'],
                "Test_ID": i + 1,
                "Prompt_Tokens": prompt_len,
                "Latenza_Sec": round(latenza_sec, 3),
                "Tokens_Per_Sec": round(tps, 2),
                "Exact_Match": 1 if passed else 0,
                "Slot_Filling_Acc": round(parziale, 3),
                "Output_Grezzo": cleaned
            })

    print(f"🧹 Pulizia VRAM...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# Salvataggio CSV
csv_save_path = os.path.join(DRIVE_FOLDER, "benchmark_completo_tesiR2.csv")
df = pd.DataFrame(risultati_globali)
df.to_csv(csv_save_path, index=False)
print(f"\n✅ DATI SALVATI IN: {csv_save_path}")

## ➕ Benchmark Aggiuntivo — Qwen3.5-4B Base (Append)
Esegue il benchmark sul modello base Qwen3.5-4B e aggiunge i risultati al CSV esistente, senza sovrascrivere i dati precedenti.

⚠️ **Esegui questa cella SOLO dopo aver completato la Cella 2.**

In [ ]:
import json
import torch
import re
import time
import pandas as pd
import gc
import os
from tqdm import tqdm
from transformers import StoppingCriteria, StoppingCriteriaList
from unsloth import FastLanguageModel
from google.colab import drive

# ==========================================
# 1. MONTAGGIO DRIVE E PERCORSI
# ==========================================
drive.mount('/content/drive')
DRIVE_FOLDER = "/content/drive/MyDrive/Tirocinio_bechelor/"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# ==========================================
# 2. FUNZIONI DI SUPPORTO
# ==========================================
def normalize(obj):
    if not isinstance(obj, dict):
        return obj
    out = {}
    for k, v in obj.items():
        if isinstance(v, str):
            v = v.strip()
            if v == "_" or (v and v[0].isupper() and v.replace("_", "").isalpha()):
                v = "__VAR__"
        out[k] = v
    return out

def fix_hallucinated_json(text):
    text = re.sub(r'([a-zA-Z0-9_]+)\s*=', r'"\1": ', text)
    text = re.sub(r'([{,]\s*)([a-zA-Z0-9_]+)(\s*:)', r'\1"\2"\3', text)
    text = re.sub(r'("arg\d+":\s*)"([^"]*)"', r'\1"\\"\2\\""', text)
    text = re.sub(r':\s*([^"\d\s\[{][^,}]*?)\s*([,}])', r': "\1"\2', text)
    return text

def calcola_accuratezza_parziale(expected_dict, predicted_dict):
    if not isinstance(expected_dict, dict) or not isinstance(predicted_dict, dict):
        return 0.0
    totale_chiavi = len(expected_dict)
    if totale_chiavi == 0:
        return 1.0 if len(predicted_dict) == 0 else 0.0
    chiavi_corrette = sum(1 for k, v in expected_dict.items() if k in predicted_dict and predicted_dict[k] == v)
    return chiavi_corrette / totale_chiavi

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set([s for s in stop_ids if s is not None])
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1].item() in self.stop_ids

# ==========================================
# 3. CONFIGURAZIONE (SOLO QWEN3.5-4B BASE)
# ==========================================
esperimenti = [
    {
        "nome_modello": "Qwen3.5-4B (Base)",
        "path": "unsloth/Qwen3.5-4B",
        "trained": "No",
        "datasets": [
            {"id": "base_ticket_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
            {"id": "base_ticket_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"},
            {"id": "base_ignoto_zero.jsonl", "domain": "In-Domain",     "esempi": "Zero-Shot"},
            {"id": "base_ignoto_few.jsonl",  "domain": "In-Domain",     "esempi": "Few-Shot"}
        ]
    }
]

# ==========================================
# 4. INFERENZA
# ==========================================
risultati_globali = []

for config_mod in esperimenti:
    print(f"\n🚀 CARICAMENTO: {config_mod['nome_modello']}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = config_mod['path'],
        max_seq_length = 4096,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

    text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
    im_end_id = text_tokenizer.convert_tokens_to_ids("<|im_end|>")
    stop_crit = StoppingCriteriaList([StopOnTokens([text_tokenizer.eos_token_id, im_end_id])])

    for ds_config in config_mod['datasets']:
        file_path_drive = os.path.join(DRIVE_FOLDER, ds_config['id'])

        if not os.path.exists(file_path_drive):
            print(f"❌ SALTO: {ds_config['id']} non trovato su Drive.")
            continue

        with open(file_path_drive, encoding="utf-8") as f:
            test_cases = [json.loads(line) for line in f if line.strip()]

        for i, tc in enumerate(tqdm(test_cases, desc=f"Test {ds_config['id']}")):
            system = tc["messages"][0]["content"]
            user = tc["messages"][1]["content"]
            expected = tc["messages"][2]["content"]

            prompt_testuale = text_tokenizer.apply_chat_template(
                [{"role": "system", "content": system}, {"role": "user", "content": user}],
                tokenize=False, add_generation_prompt=False
            )
            prompt_testuale += "<|im_start|>assistant\n{"

            tokens = text_tokenizer(prompt_testuale, return_tensors="pt")
            input_ids = tokens["input_ids"].to("cuda")
            attention_mask = tokens["attention_mask"].to("cuda") if "attention_mask" in tokens else torch.ones_like(input_ids)
            prompt_len = input_ids.shape[1]

            start_time = time.time()
            with torch.no_grad():
                output_ids = model.generate(
                    input_ids = input_ids,
                    attention_mask = attention_mask,
                    max_new_tokens = 128,
                    do_sample = False,
                    pad_token_id = text_tokenizer.eos_token_id,
                    stopping_criteria = stop_crit,
                )
            end_time = time.time()

            latenza_sec = end_time - start_time
            token_generati = len(output_ids[0]) - prompt_len
            tps = token_generati / latenza_sec if latenza_sec > 0 else 0

            predicted_raw = text_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
            predicted_raw = re.sub(r'<think>.*?</think>', '', predicted_raw, flags=re.DOTALL).strip()
            predicted_raw = "{" + predicted_raw

            cleaned = predicted_raw
            backticks = "`" * 3
            if cleaned.startswith(backticks + "json"): cleaned = cleaned[7:]
            elif cleaned.startswith(backticks): cleaned = cleaned[3:]
            if cleaned.endswith(backticks): cleaned = cleaned[:-3]
            cleaned = cleaned.strip()

            try:
                json.loads(cleaned)
            except json.JSONDecodeError:
                cleaned = fix_hallucinated_json(cleaned)

            passed = False
            parziale = 0.0
            try:
                dict_atteso = normalize(json.loads(expected))
                dict_previsto = normalize(json.loads(cleaned))
                passed = (dict_atteso == dict_previsto)
                parziale = calcola_accuratezza_parziale(dict_atteso, dict_previsto)
            except json.JSONDecodeError:
                pass

            risultati_globali.append({
                "Modello": config_mod['nome_modello'],
                "Addestrato": config_mod['trained'],
                "Dominio": ds_config['domain'],
                "Strategia": ds_config['esempi'],
                "Test_ID": i + 1,
                "Prompt_Tokens": prompt_len,
                "Latenza_Sec": round(latenza_sec, 3),
                "Tokens_Per_Sec": round(tps, 2),
                "Exact_Match": 1 if passed else 0,
                "Slot_Filling_Acc": round(parziale, 3),
                "Output_Grezzo": cleaned
            })

    print(f"🧹 Pulizia VRAM...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# Salvataggio con anti-duplicazione
csv_save_path = os.path.join(DRIVE_FOLDER, "benchmark_completo_tesiR2.csv")
nuovi_dati_df = pd.DataFrame(risultati_globali)

if os.path.exists(csv_save_path):
    df_esistente = pd.read_csv(csv_save_path)
    modelli_nuovi = nuovi_dati_df['Modello'].unique()
    righe_prima = len(df_esistente)
    df_esistente = df_esistente[~df_esistente['Modello'].isin(modelli_nuovi)]
    righe_rimosse = righe_prima - len(df_esistente)
    if righe_rimosse > 0:
        print(f"⚠️ Rimosse {righe_rimosse} righe duplicate dei modelli: {list(modelli_nuovi)}")
    df_combinato = pd.concat([df_esistente, nuovi_dati_df], ignore_index=True)
    df_combinato.to_csv(csv_save_path, index=False)
    print(f"\n✅ DATI AGGIUNTI IN APPEND AL FILE: {csv_save_path}")
else:
    nuovi_dati_df.to_csv(csv_save_path, index=False)
    print(f"\n✅ NUOVO FILE CREATO: {csv_save_path}")

## ✅ Verifica Contenuto CSV
Controlla quali modelli sono presenti nel file e mostra un'anteprima dei dati.

In [ ]:
import pandas as pd
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_FOLDER = "/content/drive/MyDrive/Tirocinio_bechelor/"
csv_save_path = os.path.join(DRIVE_FOLDER, "benchmark_completo_tesiR2.csv")

print("⏳ Caricamento CSV in corso...")
df = pd.read_csv(csv_save_path)

print("\n✅ MODELLI TROVATI NEL CSV:")
print(df['Modello'].unique())

colonne_da_mostrare = ['Modello', 'Tokens_Per_Sec', 'Exact_Match', 'Slot_Filling_Acc']
print("\n--- ANTEPRIMA DATI ---")
print(df[colonne_da_mostrare].head(20).to_string(index=False))

## 📈 Generazione Grafici (PDF)
Genera 4 grafici di confronto esportati come PDF ad alta risoluzione sul Drive.
I grafici si adattano automaticamente al numero di modelli presenti nel CSV.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

DRIVE_FOLDER = "/content/drive/MyDrive/Tirocinio_bechelor/"
csv_save_path = os.path.join(DRIVE_FOLDER, "benchmark_completo_tesiR2.csv")

if not os.path.exists(csv_save_path):
    print(f"❌ ERRORE: Manca il file CSV in {csv_save_path}. Esegui prima le celle di benchmark.")
else:
    df = pd.read_csv(csv_save_path)
    print("📊 Generazione dei grafici in corso...")

    df["Exact_Match_Pct"] = df["Exact_Match"] * 100
    df["Slot_Filling_Pct"] = df["Slot_Filling_Acc"] * 100
    agg_df = df.groupby(["Modello", "Dominio", "Strategia"])[["Exact_Match_Pct", "Slot_Filling_Pct", "Latenza_Sec"]].mean().reset_index()

    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

    # GRAFICO 1: Slot Filling
    plt.figure(figsize=(12, 6))
    ax1 = sns.barplot(data=agg_df, x="Strategia", y="Slot_Filling_Pct", hue="Modello", palette="Set1")
    for container in ax1.containers:
        ax1.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9, fontweight='bold')
    plt.title("Confronto Slot-Filling", fontweight="bold", pad=15)
    plt.ylabel("Accuratezza Parziale Media (%)")
    plt.xlabel("Strategia di Prompting")
    plt.yticks(range(0, 105, 10))
    plt.ylim(0, 115)
    plt.legend(title="Modello", bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_FOLDER, "1_Slot_Filling_Confronto.pdf"), dpi=300)
    plt.close()

    # GRAFICO 2: Exact Match
    plt.figure(figsize=(12, 6))
    ax2 = sns.barplot(data=agg_df, x="Strategia", y="Exact_Match_Pct", hue="Modello", palette="Set2")
    for container in ax2.containers:
        ax2.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9, fontweight='bold')
    plt.title("Confronto Exact Match", fontweight="bold", pad=15)
    plt.ylabel("Accuratezza Esatta Media (%)")
    plt.xlabel("Strategia di Prompting")
    plt.yticks(range(0, 105, 10))
    plt.ylim(0, 115)
    plt.legend(title="Modello", bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_FOLDER, "2_Exact_Match_Confronto.pdf"), dpi=300)
    plt.close()

    # GRAFICO 3: Heatmap dinamica per N modelli
    modelli_presenti = agg_df["Modello"].unique()
    num_modelli = len(modelli_presenti)
    fig, axes = plt.subplots(1, num_modelli, figsize=(6 * num_modelli, 5))
    if num_modelli == 1: axes = [axes]
    colori_heatmap = ["Reds", "Blues", "Greens", "Purples"]
    for i, modello in enumerate(modelli_presenti):
        pivot_modello = agg_df[agg_df["Modello"] == modello].pivot_table(
            values="Slot_Filling_Pct", index="Strategia", columns="Dominio"
        )
        if not pivot_modello.empty:
            cmap_scelta = colori_heatmap[i % len(colori_heatmap)]
            sns.heatmap(pivot_modello, annot=True, fmt=".2f", cmap=cmap_scelta,
                        vmin=0, vmax=100, ax=axes[i], annot_kws={"weight": "bold"})
            axes[i].set_title(f"{modello}", fontweight="bold", fontsize=11)
    plt.suptitle("Generalizzazione sui Domini (Slot-Filling %)", fontweight="bold", fontsize=15)
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_FOLDER, "3_Heatmap_Confronto.pdf"), dpi=300)
    plt.close()

    # GRAFICO 4: Latenza
    plt.figure(figsize=(12, 6))
    ax3 = sns.barplot(data=agg_df, x="Modello", y="Latenza_Sec", hue="Strategia", palette="magma")
    for container in ax3.containers:
        ax3.bar_label(container, fmt='%.2f s', padding=3, fontsize=9)
    plt.title("Tempo di Generazione per Query", fontweight="bold", pad=15)
    plt.ylabel("Secondi medi per query")
    plt.xlabel("Configurazione Modello")
    plt.xticks(rotation=15)
    massimo_latenza = agg_df["Latenza_Sec"].max()
    plt.ylim(0, massimo_latenza * 1.20)
    plt.legend(title="Strategia")
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_FOLDER, "4_Latenza_Confronto.pdf"), dpi=300)
    plt.close()

    print("✅ TUTTO COMPLETATO! I 4 PDF sono salvati sul Drive.")

## 🔍 Ispezione Visiva (Opzionale)
Stampa a schermo input, output atteso e output del modello per ogni test case del modello Base.
Utile per capire *come* il modello sbaglia.

In [ ]:
import json
import torch
import re
import os
from transformers import StoppingCriteria, StoppingCriteriaList
from unsloth import FastLanguageModel
from google.colab import drive

drive.mount('/content/drive')
DRIVE_FOLDER = "/content/drive/MyDrive/Tirocinio_bechelor/"

def normalize(obj):
    if not isinstance(obj, dict):
        return obj
    out = {}
    for k, v in obj.items():
        if isinstance(v, str):
            v = v.strip()
            if v == "_" or (v and v[0].isupper() and v.replace("_", "").isalpha()):
                v = "__VAR__"
        out[k] = v
    return out

def fix_hallucinated_json(text):
    text = re.sub(r'([a-zA-Z0-9_]+)\s*=', r'"\1": ', text)
    text = re.sub(r'([{,]\s*)([a-zA-Z0-9_]+)(\s*:)', r'\1"\2"\3', text)
    text = re.sub(r'("arg\d+":\s*)"([^"]*)"', r'\1"\\"\2\\""', text)
    text = re.sub(r':\s*([^"\d\s\[{][^,}]*?)\s*([,}])', r': "\1"\2', text)
    return text

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set([s for s in stop_ids if s is not None])
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1].item() in self.stop_ids

config_mod = {
    "nome_modello": "Qwen2.5-Coder-7B (Base)",
    "path": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    "datasets": [
        {"id": "base_ticket_zero.jsonl", "domain": "Out-of-Domain", "esempi": "Zero-Shot"},
        {"id": "base_ticket_few.jsonl",  "domain": "Out-of-Domain", "esempi": "Few-Shot"},
        {"id": "base_ignoto_zero.jsonl", "domain": "In-Domain",     "esempi": "Zero-Shot"},
        {"id": "base_ignoto_few.jsonl",  "domain": "In-Domain",     "esempi": "Few-Shot"}
    ]
}

print(f"\n🚀 CARICAMENTO: {config_mod['nome_modello']}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config_mod['path'],
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
im_end_id = text_tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_crit = StoppingCriteriaList([StopOnTokens([text_tokenizer.eos_token_id, im_end_id])])

for ds_config in config_mod['datasets']:
    file_path_drive = os.path.join(DRIVE_FOLDER, ds_config['id'])
    if not os.path.exists(file_path_drive):
        print(f"❌ SALTO: {ds_config['id']} non trovato su Drive.")
        continue
    with open(file_path_drive, encoding="utf-8") as f:
        test_cases = [json.loads(line) for line in f if line.strip()]
    print(f"\n{'='*60}")
    print(f"📊 DATASET: {ds_config['id']}")
    print(f"{'='*60}")
    for i, tc in enumerate(test_cases):
        system = tc["messages"][0]["content"]
        user = tc["messages"][1]["content"]
        expected = tc["messages"][2]["content"]
        prompt_testuale = text_tokenizer.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=False
        )
        prompt_testuale += "<|im_start|>assistant\n{"
        tokens = text_tokenizer(prompt_testuale, return_tensors="pt")
        input_ids = tokens["input_ids"].to("cuda")
        attention_mask = tokens["attention_mask"].to("cuda") if "attention_mask" in tokens else torch.ones_like(input_ids)
        prompt_len = input_ids.shape[1]
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids, attention_mask=attention_mask,
                max_new_tokens=128, do_sample=False,
                pad_token_id=text_tokenizer.eos_token_id, stopping_criteria=stop_crit,
            )
        predicted_raw = text_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
        predicted_raw = re.sub(r'<think>.*?</think>', '', predicted_raw, flags=re.DOTALL).strip()
        predicted_raw = "{" + predicted_raw
        cleaned = predicted_raw
        backticks = "`" * 3
        if cleaned.startswith(backticks + "json"): cleaned = cleaned[7:]
        elif cleaned.startswith(backticks): cleaned = cleaned[3:]
        if cleaned.endswith(backticks): cleaned = cleaned[:-3]
        cleaned = cleaned.strip()
        try:
            json.loads(cleaned)
        except json.JSONDecodeError:
            cleaned = fix_hallucinated_json(cleaned)
        passed = False
        try:
            passed = normalize(json.loads(cleaned)) == normalize(json.loads(expected))
        except json.JSONDecodeError:
            pass
        print(f"\n--- Test {i+1}/{len(test_cases)} ---")
        print(f"🗣️ INPUT:   {user}")
        print(f"✅ ATTESO:  {expected}")
        print(f"🤖 OUTPUT:  {cleaned}")
        print(f"{'🟢 MATCH' if passed else '🔴 FAIL'}")
        print("-" * 40)

print(f"\n🧹 Pulizia VRAM...")
del model
del tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()